# Cats vs Dogs Binary Classification - Image Classification with Data Augmentation

Week 11, Day 3 - Mini-project

**Roadmap:** load the images and build Keras generators → inspect class balance and samples → design a small CNN → train with and without augmentation and compare → evaluate on a held-out validation split → run inference on the unlabeled test folder and export predictions → save the model and its configuration.

> **Resolution:** the original images are large (up to 500×500). As suggested, we reduce `IMG_HEIGHT` and `IMG_WIDTH` to **64×64** so that each epoch is fast enough on a CPU/VM, while keeping enough detail to see ears, eyes and muzzles.

## 0. Download the dataset

The zip is downloaded from the bootcamp repository, extracted, and the folder is renamed `cats_dogs` inside a `data` folder, as required.

In [ ]:
import os, zipfile, urllib.request, shutil
from pathlib import Path

ZIP_PATH = Path("Dogs vs Cats.zip")
ZIP_URL = ("https://media.githubusercontent.com/media/devtlv/Datasets-GEN-AI-Bootcamp/main/"
           "Week%206/W6D5/Dogs%20vs%20Cats.zip")
TARGET = Path("data/cats_dogs")

if not TARGET.exists():
    if not ZIP_PATH.exists():
        print("Downloading dataset (~860 MB)...")
        urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    tmp = Path("data/_extract")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(tmp)
    # The zip contains train/ and test/ at its root (possibly inside one top folder): move them to data/cats_dogs
    inner = tmp if (tmp / "train").exists() else next(p for p in tmp.iterdir() if p.is_dir())
    shutil.move(str(inner), str(TARGET)) if inner != tmp else tmp.rename(TARGET)
    shutil.rmtree(tmp, ignore_errors=True)
print("Dataset folder:", TARGET, "->", sorted(p.name for p in TARGET.iterdir()))

## 1. Data loading and generators

*Prefilled code, copied and executed (only `IMG_HEIGHT` and `IMG_WIDTH` were reduced to 64, and the TensorFlow logs are silenced).*

In [ ]:
# Prefilled. Just copy and execute.
import os, math, re, random
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

np.random.seed(42); tf.random.set_seed(42)

# Paths - change if needed
DATA_ROOT = Path("data/cats_dogs")
train_dir = (DATA_ROOT / "train" / "train") if (DATA_ROOT / "train" / "train").exists() else (DATA_ROOT / "train")
test_dir  = (DATA_ROOT / "test"  / "test")  if (DATA_ROOT / "test"  / "test").exists()  else (DATA_ROOT / "test")

IMG_HEIGHT, IMG_WIDTH = 64, 64   # reduced from 180x180 for faster training
batch_size = 32
seed = 1337

# Build DataFrames from folders
def build_df_from_folder(folder: Path, labeled: bool=True):
    exts = ('*.jpg','*.jpeg','*.png','*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder / '**' / ex), recursive=True))
    if not files:
        raise FileNotFoundError(f"No images found under {folder}")
    rows = []
    for f in files:
        if labeled:
            name = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {"cat","cats"}:
                label = "cat"
            elif parent in {"dog","dogs"}:
                label = "dog"
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name): label = "cat"
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name): label = "dog"
                else:
                    continue
            rows.append({"filepath": f, "label": label})
        else:
            rows.append({"filepath": f})
    return pd.DataFrame(rows)

df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full  = build_df_from_folder(test_dir,  labeled=False)

# Train validation split
from sklearn.model_selection import train_test_split
df_tr, df_val = train_test_split(
    df_train_full, test_size=0.2, stratify=df_train_full["label"], random_state=seed
)

# Generators
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.5,
    horizontal_flip=True,
)
val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    df_tr, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=True, seed=seed, validate_filenames=False
)
val_flow = val_gen.flow_from_dataframe(
    df_val, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=False, validate_filenames=False
)
# Unlabeled test for inference only
test_flow = test_gen.flow_from_dataframe(
    df_test_full, x_col="filepath", y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None, batch_size=batch_size,
    shuffle=False, validate_filenames=False
)

print({"train": train_flow.samples, "val": val_flow.samples, "test": test_flow.samples,
       "class_indices": train_flow.class_indices})

## 2. Inspect the data

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

idx_to_class = {v: k for k, v in train_flow.class_indices.items()}
train_counts = pd.Series(train_flow.labels).map(idx_to_class).value_counts()
val_counts = pd.Series(val_flow.labels).map(idx_to_class).value_counts()
counts = pd.DataFrame({'train': train_counts, 'validation': val_counts, 'full labeled set': df_train_full['label'].value_counts()})
display(counts)
print("Class indices:", train_flow.class_indices, "-> the sigmoid output is P(dog)")

sizes = pd.DataFrame([Image.open(f).size for f in df_train_full['filepath'].sample(500, random_state=seed)], columns=['width', 'height'])
print("Original image sizes (sample of 500):")
display(sizes.describe().round(0))

In [ ]:
# Grid of training images (original files) annotated with their label
sample = df_tr.sample(16, random_state=seed)
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    ax.imshow(Image.open(row['filepath']))
    ax.set_title(row['label'], color='tab:orange' if row['label'] == 'cat' else 'tab:blue', fontsize=12)
    ax.axis('off')
fig.suptitle('Training images and their labels (original resolution)')
plt.tight_layout(); plt.show()

# The same kind of images as the model sees them: 64x64, augmented
xb, yb = next(train_flow)
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for ax, img, lab in zip(axes.ravel(), xb, yb):
    ax.imshow(img)
    ax.set_title(idx_to_class[int(lab)], fontsize=12)
    ax.axis('off')
fig.suptitle('One augmented training batch as seen by the model (64x64, rotation / shift / zoom / flip)')
plt.tight_layout(); plt.show()

**Data report:**
- The labeled set has **25,000 images: 12,500 cats and 12,500 dogs**. After the stratified 80/20 split, the training generator has **20,000 images (10,000 per class)** and the validation generator **5,000 images (2,500 per class)**. `class_indices` maps `cat → 0` and `dog → 1`, so the sigmoid output of the model is the probability of *dog*.
- The classes are **perfectly balanced**, so accuracy is a meaningful metric and no class weights or resampling are needed (see Part 9).
- The **12,500 test images** are unlabeled and only used for the final predictions.
- **Sources of visual variability:** very different **poses** (sitting, lying, jumping, only the head visible), **scales** (close-up portrait vs. small animal in a large scene), **lighting** (flash, outdoor sun, dark rooms), **backgrounds** (sofas, grass, cages, people holding the animal), many **breeds and colours** (some dogs are small and fluffy like cats, some cats have dog-like colours), image sizes and aspect ratios (from about 50 to 500 pixels), and sometimes several animals, text or borders in the same picture.

**Visual cues that should help the model:** the **shape of the face** (cats have a short, round face with a small nose; dogs usually have a longer muzzle and a bigger, darker nose), the **ears** (cats have triangular, upright ears; many dogs have floppy or larger ears), the **eyes** (cats' eyes are larger relative to the face, often with slit pupils), whiskers, and the general body proportions. Fur colour and background are much less reliable: a model relying on them would learn biases (e.g. "grass = dog").

## 3. Define the model architecture

**Planned CNN:**
- **Three convolutional blocks** with **32, 64 and 128 filters** of size **3×3** and ReLU activation. The number of filters doubles at each block: the first layers detect simple local patterns (edges, colour contrasts, fur texture), the deeper ones combine them into more complex parts (eyes, ears, nose).
- A **2×2 MaxPooling** after each convolution, which halves the spatial size (64 → 31 → 14 → 6). Pooling reduces the number of parameters and the computation, and makes the detection of a pattern tolerant to small shifts (translational invariance).
- After **Flatten**, a **Dropout(0.5)** layer randomly switches off half of the features at each training step. This prevents the network from relying on a few specific pixels or co-adapted neurons, i.e. it regularises the model and reduces overfitting.
- A hidden **Dense(128, ReLU)** layer combines the features, followed by the output **Dense(1, sigmoid)** unit, which gives the probability that the image is a dog.

**Loss:** binary cross-entropy, the correct loss for a Bernoulli target (cat = 0 / dog = 1) with a sigmoid output.

In [ ]:
from tensorflow.keras import layers, models

def build_model(dropout=0.5, learning_rate=1e-3):
    model = models.Sequential([
        layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dropout(dropout),
        layers.Dense(128, activation='relu'),
        layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = build_model()
model.summary()

## 4. Choose the optimization setup

- **Optimizer: Adam.** It adapts the learning rate of each weight using running averages of the gradients and of their squares, so it converges quickly without much tuning, which is ideal for a first CNN.
- **Initial learning rate: 1e-3**, Adam's default. It is large enough to learn quickly in the first epochs and small enough to stay stable with batch-sized gradient estimates; if the validation loss plateaus, it will be reduced automatically.
- **Batch size: 32.** At 64×64×3 pixels, a batch takes less than 2 MB of input data plus the activations, which fits easily in CPU or GPU memory, and 32 gives a good balance between noisy (regularising) gradients and speed.
- **EarlyStopping** on the **validation loss** (patience 4, best weights restored): training stops when the validation loss has not improved for 4 epochs, before the model starts overfitting.
- **ReduceLROnPlateau** on the validation loss (factor 0.5, patience 2): when the validation loss stops improving, the learning rate is halved, which helps the model settle into a better minimum.

We monitor both the **loss** (smoother and more sensitive to the quality of the probabilities) and the **accuracy** (easier to interpret; reliable here since the classes are balanced).

## 5. Train the model

### 5.1 Fixed number of epochs (with augmentation)

In [ ]:
import time, json

def plot_history(histories, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for name, h in histories.items():
        ep = range(1, len(h['loss']) + 1)
        axes[0].plot(ep, h['loss'], '--', label=f'{name} - train')
        axes[0].plot(ep, h['val_loss'], 'o-', label=f'{name} - validation')
        axes[1].plot(ep, h['accuracy'], '--', label=f'{name} - train')
        axes[1].plot(ep, h['val_accuracy'], 'o-', label=f'{name} - validation')
    axes[0].set_title(f'{title} - loss'); axes[0].set_xlabel('epoch'); axes[0].legend(fontsize=8)
    axes[1].set_title(f'{title} - accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()

run_log = {}

tf.keras.utils.set_random_seed(42)
model_fixed = build_model()
t0 = time.time()
hist_fixed = model_fixed.fit(train_flow, validation_data=val_flow, epochs=12, verbose=2)
run_log['augmented_fixed_12_epochs'] = {'epochs': 12, 'time_s': round(time.time() - t0)}
plot_history({'augmented, 12 epochs': hist_fixed.history}, 'Fixed number of epochs')

### 5.2 Same model with EarlyStopping and ReduceLROnPlateau

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
os.makedirs("artifacts", exist_ok=True)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1),
    ModelCheckpoint('artifacts/best_model.keras', monitor='val_loss', save_best_only=True),
]

tf.keras.utils.set_random_seed(42)
model_aug = build_model()
t0 = time.time()
hist_aug = model_aug.fit(train_flow, validation_data=val_flow, epochs=20, callbacks=callbacks, verbose=2)
run_log['augmented_early_stopping'] = {'epochs_run': len(hist_aug.history['loss']), 'time_s': round(time.time() - t0),
                                       'best_epoch': int(np.argmin(hist_aug.history['val_loss'])) + 1}
plot_history({'augmented + early stopping': hist_aug.history}, 'With EarlyStopping / ReduceLROnPlateau')
print(run_log)

TRAIN_PLACEHOLDER

## 6. Evaluate on validation data

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

val_loss, val_acc = model_aug.evaluate(val_flow, verbose=0)
print(f"Validation loss: {val_loss:.4f} | validation accuracy: {val_acc:.4f}")

val_prob = model_aug.predict(val_flow, verbose=0).ravel()   # P(dog); val_flow is not shuffled
y_val = np.array(val_flow.labels)
val_pred = (val_prob >= 0.5).astype(int)

print(classification_report(y_val, val_pred, target_names=['cat', 'dog'], digits=4))
cm = confusion_matrix(y_val, val_pred)
ConfusionMatrixDisplay(cm, display_labels=['cat', 'dog']).plot(cmap='Blues', colorbar=False)
plt.title('Validation confusion matrix (threshold 0.5)')
plt.show()
print(f"Cats predicted as dogs: {cm[0, 1]} | dogs predicted as cats: {cm[1, 0]}")

In [ ]:
# Effect of the decision threshold
rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (val_prob >= t).astype(int)
    c = confusion_matrix(y_val, p)
    rows.append({'threshold': t, 'accuracy': (p == y_val).mean(), 'cats -> dog': c[0, 1], 'dogs -> cat': c[1, 0]})
thr_df = pd.DataFrame(rows).set_index('threshold')
display(thr_df.style.format({'accuracy': '{:.4f}'}))

# Some misclassified validation images
wrong = np.where(val_pred != y_val)[0]
order = wrong[np.argsort(-np.abs(val_prob[wrong] - 0.5))][:12]  # most confident mistakes
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, k in zip(axes.ravel(), order):
    ax.imshow(Image.open(df_val['filepath'].iloc[k]))
    ax.set_title(f"true {idx_to_class[y_val[k]]} | P(dog)={val_prob[k]:.2f}", fontsize=10, color='red')
    ax.axis('off')
fig.suptitle('Most confident mistakes on the validation set')
plt.tight_layout(); plt.show()

EVAL_PLACEHOLDER

## 7. Run inference on the unlabeled test set

In [ ]:
THRESHOLD = 0.5   # balanced classes and balanced errors on validation -> keep 0.5 (see Part 6)

test_flow.reset()
test_prob = model_aug.predict(test_flow, verbose=0).ravel()
predictions = pd.DataFrame({
    'filepath': df_test_full['filepath'].values,
    'prob_dog': test_prob.round(5),
    'pred_label': np.where(test_prob >= THRESHOLD, 'dog', 'cat'),
})
predictions.to_csv('test_predictions.csv', index=False)
print(predictions.shape)
display(predictions.head())
print(predictions['pred_label'].value_counts(normalize=True).round(3))

plt.figure(figsize=(7, 3))
plt.hist(test_prob, bins=50)
plt.axvline(THRESHOLD, color='red', ls='--')
plt.title('Distribution of P(dog) on the test set'); plt.xlabel('P(dog)')
plt.show()

In [ ]:
# Manual sanity check: random test images with their prediction
check = predictions.sample(12, random_state=0)
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, (_, row) in zip(axes.ravel(), check.iterrows()):
    ax.imshow(Image.open(row['filepath']))
    ax.set_title(f"{row['pred_label']} ({row['prob_dog']:.2f})", fontsize=11)
    ax.axis('off')
fig.suptitle('Random test images with predicted label and P(dog)')
plt.tight_layout(); plt.show()

TEST_PLACEHOLDER

## 8. Compare baseline vs augmentation

Same architecture, same optimisation setup and callbacks, but the training generator only rescales the images (no rotation, shift, zoom or flip).

In [ ]:
plain_gen = ImageDataGenerator(rescale=1./255)
train_flow_plain = plain_gen.flow_from_dataframe(
    df_tr, x_col="filepath", y_col="label", target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size, shuffle=True, seed=seed, validate_filenames=False)

tf.keras.utils.set_random_seed(42)
model_base = build_model()
t0 = time.time()
hist_base = model_base.fit(train_flow_plain, validation_data=val_flow, epochs=20, verbose=2, callbacks=[
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)])
run_log['baseline_no_augmentation'] = {'epochs_run': len(hist_base.history['loss']), 'time_s': round(time.time() - t0),
                                       'best_epoch': int(np.argmin(hist_base.history['val_loss'])) + 1}
plot_history({'no augmentation': hist_base.history, 'augmentation': hist_aug.history}, 'Baseline vs augmentation')

In [ ]:
def summary_row(h, model):
    i = int(np.argmin(h['val_loss']))
    loss, acc = model.evaluate(val_flow, verbose=0)
    return {'best epoch': i + 1, 'train acc (best epoch)': h['accuracy'][i], 'val acc': acc, 'val loss': loss,
            'generalization gap': h['accuracy'][i] - acc}

compare = pd.DataFrame({'no augmentation': summary_row(hist_base.history, model_base),
                        'augmentation': summary_row(hist_aug.history, model_aug)}).T
display(compare.style.format({'best epoch': '{:.0f}', 'train acc (best epoch)': '{:.4f}', 'val acc': '{:.4f}',
                              'val loss': '{:.4f}', 'generalization gap': '{:+.4f}'}))

COMPARE_PLACEHOLDER

## 9. Class imbalance handling

The classes are **perfectly balanced** (10,000 cats and 10,000 dogs in the training split), so the balanced class weights computed below are exactly 1.0 for both classes: using `class_weight` would give the same loss as without it, so retraining is not necessary.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=np.array(train_flow.labels))
class_weight = {i: float(w) for i, w in enumerate(weights)}
print("Balanced class weights:", class_weight)
# If they were different, we would retrain with:
# model.fit(train_flow, validation_data=val_flow, epochs=..., class_weight=class_weight)

If one class were rare (e.g. 20% cats), its errors would only count for a small part of the loss, so the network would favour the majority class: high recall for dogs, low recall for cats. With `class_weight`, each cat image would count more (weight = n_samples / (2 × n_cats)), which **increases the recall of the minority class** (more cats found), usually at the cost of a slightly lower **precision** for that class (some dogs predicted as cats).

## 10. Save artifacts for reuse

In [ ]:
model_aug.save('artifacts/cats_dogs_cnn.h5')        # HDF5 format
# artifacts/best_model.keras was already saved by ModelCheckpoint during training (native Keras format)

config = {
    'image_size': [IMG_HEIGHT, IMG_WIDTH], 'batch_size': batch_size, 'seed': seed,
    'class_indices': train_flow.class_indices, 'output': 'sigmoid = P(dog)', 'threshold': THRESHOLD,
    'split': {'train': train_flow.samples, 'validation': val_flow.samples, 'test': test_flow.samples},
    'augmentation': {'rotation_range': 45, 'width_shift_range': 0.15, 'height_shift_range': 0.15,
                     'zoom_range': 0.5, 'horizontal_flip': True, 'rescale': '1/255'},
    'architecture': 'Conv32-Pool-Conv64-Pool-Conv128-Pool-Flatten-Dropout0.5-Dense128-Dense1(sigmoid)',
    'optimizer': {'name': 'adam', 'learning_rate': 1e-3},
    'callbacks': ['EarlyStopping(val_loss, patience=4)', 'ReduceLROnPlateau(val_loss, factor=0.5, patience=2)'],
    'results': {'val_accuracy': float(val_acc), 'val_loss': float(val_loss)},
    'runs': run_log,
    'tensorflow_version': tf.__version__,
}
with open('artifacts/training_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

reloaded = tf.keras.models.load_model('artifacts/cats_dogs_cnn.h5')
print("Reloaded model - validation accuracy:", round(reloaded.evaluate(val_flow, verbose=0)[1], 4))
print("Saved files:", os.listdir('artifacts'))

**Why save both the weights and the metadata?** The model file alone is not enough to reproduce the results or to use the model correctly: we also need to know the **input size** (64×64), the **preprocessing** (pixels divided by 255), the **meaning of the output** (`cat = 0`, `dog = 1`, so the sigmoid gives P(dog)) and the **threshold**. Recording the augmentation, the optimizer, the callbacks, the data split and the library version makes it possible to retrain the same model later, compare new experiments fairly, and audit how a deployed model was produced.

## 11. Extensions

**Proposed extension: transfer learning with a frozen MobileNetV2 backbone.** MobileNetV2 was pre-trained on ImageNet (1.2 million images, including many cat and dog breeds), so its convolutional layers already detect edges, fur textures, eyes, ears and muzzles. We would use it as a frozen feature extractor (`include_top=False`, input at least 96×96 or 128×128), add a small head (`GlobalAveragePooling2D → Dropout → Dense(1, sigmoid)`) and train only this head, then optionally fine-tune the last blocks with a small learning rate.

**Expected benefit:** a much higher accuracy (typically more than 95% on Cats vs Dogs, against about 80–85% for our small CNN trained from scratch at 64×64), in fewer epochs, because the network does not need to learn low-level visual features from only 20,000 images. MobileNetV2 is lightweight (about 2.3 million parameters), so it stays fast enough for a CPU/VM. **Batch Normalization** after the convolutions would also stabilise and speed up the training of our own CNN, and **mixed precision** would mainly speed up training on a modern GPU, without improving the accuracy.

FINAL_PLACEHOLDER